# Taller II — Integración de fuentes heterogéneas con PySpark

**Curso:** Gestión de datos y tecnologías Big Data (MCDI502) · Semana 3 · Sumativa 3
**Grupo:** 1
**Integrantes:** Yerko Gallardo, Sebastián Rojas, José Ignacio Rodríguez

**Objetivo:** capturar 100 registros de cuatro fuentes heterogéneas (dos API públicas y dos bases de datos de sensores ESP32), limpiarlos y transformarlos con Spark, integrarlos en un único DataFrame y responder consultas con Spark SQL.

| Punto de la consigna | Sección | Responsable |
|---|---|---|
| 1. Configuración inicial del entorno | 1 | Sebastián · José |
| 2. Ingesta de datos y creación de DataFrames | 2.1 OpenMeteo · 2.2 AQICN · 2.3 MongoDB · 2.4 MySQL | José · Sebastián |
| 3. Preprocesamiento y transformación | 3.1 `altura_metros` · 3.2 `condicion_clima` | Sebastián · José |
| 4. Integración de datos | 4 | Yerko |
| 5. Consultas y extracción (C1–C7) | 5 | Yerko |
| 6. Presentación | 6 (video aparte) | Todos |
| Fuentes y atribuciones | 7 | Todos |

**Ejecución:** *Reiniciar y ejecutar todo*. El notebook funciona en Google Colab y en local (VS Code con el entorno uv del repositorio). Las credenciales (URI de MongoDB y token de AQICN) no están en el código: se leen de variables de entorno, de archivos privados en `.local/` o se piden con una entrada oculta (sección 1.3).

## 1. Configuración inicial del entorno

| Componente | Versión | Motivo |
|---|---|---|
| Python | 3.14 en local (uv) · la de Colab | Fijada en `.python-version` |
| PySpark (incluye Apache Spark) | 4.3.0 | Fijada en `uv.lock`; se instala si falta o difiere |
| Java (JDK) | 17 o superior | Spark 4 exige Java 17 |
| PyMongo · requests | según `uv.lock` | Conector de MongoDB y cliente HTTP para las API y el endpoint PHP |

**Decisiones respecto de los notebooks base del docente:**

- Los ejemplos usan Spark 3.5.9 con Java 8. Aquí se usa Spark 4.3.0 con Java 17, que es la versión del proyecto. Así, las cuatro fuentes comparten una sola sesión.
- MongoDB se lee con **PyMongo**, uno de los conectores que indica el enunciado. No se usa `mongo-spark-connector_2.12:3.0.1` porque está compilado para Scala 2.12 y Spark 3, y no funciona con Spark 4 (Scala 2.13). Además, PyMongo permite pedir solo los 100 documentos más recientes, en lugar de cargar la colección completa (~100 000 documentos).
- MySQL se consulta mediante el endpoint PHP/REST del curso (`obtener_datos.php`), como en el notebook base. Por eso no se necesita `mysql-connector-python` ni un controlador JDBC.

### 1.1 Instalación de Spark, Java y librerías, y variables de entorno

- **Spark:** se instala como el paquete `pyspark==4.3.0`. Si ya está en esa versión (entorno uv), no se reinstala. En Colab reemplaza la versión preinstalada.
- **Librerías:** `pymongo` y `requests` se instalan solo si faltan.
- **Java:** se reutiliza el primer JDK 17 o superior que se encuentre (`JAVA_HOME`, `java` del PATH o rutas habituales). Si no hay ninguno, se instala OpenJDK 17 con `apt-get` en Colab o Linux con root, o Temurin 17 con `install-jdk`, que no requiere administrador.
- **Variables:** `JAVA_HOME` (y su `bin` en el PATH), `PYSPARK_PYTHON`, `PYSPARK_DRIVER_PYTHON` y `SPARK_LOCAL_IP`.

In [1]:
import os, sys, platform, shutil, subprocess, re, glob, importlib.util, importlib.metadata
from pathlib import Path

VERSION_PYSPARK = "4.3.0"
JAVA_MINIMA = 17
SISTEMA = platform.system()
EJECUTABLE_JAVA = "java.exe" if SISTEMA == "Windows" else "java"
EN_COLAB = "google.colab" in sys.modules


def instalar_paquete(paquete):
    """Instala un paquete en el intérprete actual, con pip o, en entornos uv sin pip, con uv."""
    if importlib.util.find_spec("pip"):
        comando = [sys.executable, "-m", "pip", "install", "-q", paquete]
    elif shutil.which("uv"):
        comando = ["uv", "pip", "install", "-q", "--python", sys.executable, paquete]
    else:
        raise RuntimeError(f"No hay pip ni uv para instalar {paquete}")
    subprocess.run(comando, check=True)


def version_instalada(distribucion):
    try:
        return importlib.metadata.version(distribucion)
    except importlib.metadata.PackageNotFoundError:
        return None


# --- Spark y librerías de conexión ---------------------------------------
if version_instalada("pyspark") != VERSION_PYSPARK:
    print(f"Se instala pyspark=={VERSION_PYSPARK} (versión actual: {version_instalada('pyspark')})")
    instalar_paquete(f"pyspark=={VERSION_PYSPARK}")
for paquete in ["pymongo", "requests"]:
    if version_instalada(paquete) is None:
        print("Se instala", paquete)
        instalar_paquete(paquete)
print("Paquetes:", {p: version_instalada(p) for p in ["pyspark", "pymongo", "requests"]})


# --- Java ----------------------------------------------------------------
def version_java(java_home):
    """Versión mayor del JDK en java_home, o None si no hay un java ejecutable."""
    java = Path(java_home) / "bin" / EJECUTABLE_JAVA
    if not java.is_file():
        return None
    try:
        salida = subprocess.run([str(java), "-version"], capture_output=True, text=True, timeout=60).stderr
    except (OSError, subprocess.SubprocessError):
        return None
    coincidencia = re.search(r'version "(\d+)(?:\.(\d+))?', salida)
    if not coincidencia:
        return None
    mayor = int(coincidencia.group(1))
    return int(coincidencia.group(2) or 0) if mayor == 1 else mayor   # "1.8" corresponde a Java 8


def home_de_jdk(ruta):
    """En macOS el JDK descomprimido guarda su home en Contents/Home."""
    ruta = Path(ruta)
    return ruta / "Contents" / "Home" if (ruta / "Contents" / "Home").is_dir() else ruta


def candidatos_java():
    if os.environ.get("JAVA_HOME"):
        yield "JAVA_HOME", os.environ["JAVA_HOME"]
    if SISTEMA == "Darwin":
        consulta = subprocess.run(["/usr/libexec/java_home", "-v", f"{JAVA_MINIMA}+"], capture_output=True, text=True)
        if consulta.returncode == 0:
            yield "java_home de macOS", consulta.stdout.strip()
    java_en_path = shutil.which("java")
    if java_en_path:
        home = Path(os.path.realpath(java_en_path)).parent.parent
        if not (SISTEMA == "Darwin" and str(home) == "/usr"):   # lanzador del sistema, no un JDK
            yield "PATH", home
    patrones = {
        "Windows": [r"C:\Program Files\Eclipse Adoptium\*", r"C:\Program Files\Java\*", r"C:\Program Files\Microsoft\jdk-*"],
        "Darwin": ["/Library/Java/JavaVirtualMachines/*"],
        "Linux": ["/usr/lib/jvm/*"],
    }.get(SISTEMA, []) + [str(Path.home() / ".jdk" / "*")]   # instalaciones de install-jdk
    for patron in patrones:
        for ruta in sorted(glob.glob(patron), reverse=True):
            yield "ruta habitual", ruta


def seleccionar_java():
    revisados = set()
    for origen, ruta in candidatos_java():
        java_home = home_de_jdk(os.path.realpath(ruta))
        if java_home in revisados or (SISTEMA == "Darwin" and str(java_home) == "/usr"):
            continue
        revisados.add(java_home)
        version = version_java(java_home)
        if version is not None and version >= JAVA_MINIMA:
            return origen, str(java_home), version
        print(f"  Descartado ({origen}): {java_home} -> versión {version}")
    return None


def instalar_java():
    if SISTEMA == "Linux" and shutil.which("apt-get") and os.geteuid() == 0:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", f"openjdk-{JAVA_MINIMA}-jdk-headless"], check=True)
    else:
        if importlib.util.find_spec("jdk") is None:
            instalar_paquete("install-jdk")
        import jdk
        print("  Temurin instalado en:", jdk.install(str(JAVA_MINIMA)))


seleccion = seleccionar_java()
if seleccion is None:
    print(f"No se encontró Java {JAVA_MINIMA} o superior: se instala un JDK {JAVA_MINIMA}.")
    try:
        instalar_java()
    except Exception as error:
        raise RuntimeError(f"No fue posible instalar Java automáticamente ({error}). Instale Temurin "
                           f"{JAVA_MINIMA} desde https://adoptium.net y defina JAVA_HOME.") from error
    seleccion = seleccionar_java()
    if seleccion is None:
        raise RuntimeError("Java se instaló, pero no se encontró un JDK utilizable.")

origen_java, java_home, version_jdk = seleccion
print(f"Java {version_jdk} encontrado mediante {origen_java}.")

# --- Variables de entorno ------------------------------------------------
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = str(Path(java_home) / "bin") + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
for variable in ["JAVA_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON", "SPARK_LOCAL_IP"]:
    print(f"{variable:22s}= {os.environ[variable]}")

Paquetes: {'pyspark': '4.3.0', 'pymongo': '4.19.0', 'requests': '2.34.2'}
Java 17 encontrado mediante JAVA_HOME.
JAVA_HOME             = C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot
PYSPARK_PYTHON        = C:\DEV\magister\3. Gestion de Datos y Tecnologias Big Data\gestiondatosytecnologiasbigdata\.venv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON = C:\DEV\magister\3. Gestion de Datos y Tecnologias Big Data\gestiondatosytecnologiasbigdata\.venv\Scripts\python.exe
SPARK_LOCAL_IP        = 127.0.0.1


### 1.2 SparkSession y SparkContext

La sesión se crea en modo local clásico, con la zona horaria `America/Santiago` para que las fechas se muestren en hora de Chile. El `SparkContext` se obtiene de la misma sesión. Una prueba mínima confirma que el motor responde antes de conectar las fuentes.

In [2]:
import pyspark
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .master("local[2]")
         .appName("mcdi502_s3_grupo1")
         .config("spark.sql.session.timeZone", "America/Santiago")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Python      :", platform.python_version())
print("PySpark     :", pyspark.__version__)
print("Spark       :", spark.version)
print("Java        :", sc._jvm.java.lang.System.getProperty("java.version"))
print("Master      :", sc.master, "· aplicación:", sc.appName)
print("Zona horaria:", spark.conf.get("spark.sql.session.timeZone"))
assert spark.range(10).count() == 10
print("Arranque de Spark verificado")

Python      : 3.14.4
PySpark     : 4.3.0
Spark       : 4.3.0
Java        : 17.0.21
Master      : local[2] · aplicación: mcdi502_s3_grupo1
Zona horaria: America/Santiago


Arranque de Spark verificado


### 1.3 Credenciales y parámetros comunes

Las credenciales se buscan en este orden: variable de entorno (`MONGODB_URI`, `AQICN_TOKEN`), archivo privado en la carpeta `.local/` del taller (junto a `notebooks/`) (`mongodb.json` con `{"uri": ...}` y `aqicn.json` con `{"token": ...}`) y, si no hay ninguno, entrada oculta. `.local/` está excluido de Git y del ZIP. Nunca se imprimen.

Ambas API usan la misma ciudad, **Santiago de Chile**, y las fechas se interpretan en `America/Santiago`.

In [3]:
import json, time, math
from getpass import getpass
from datetime import datetime, timezone
import requests

# Carpeta del taller (la que contiene notebooks/ y .local/): funciona si Jupyter se inicia
# en notebooks/, en la carpeta del taller o en la raíz del repositorio.
NOMBRE_NOTEBOOK = Path("notebooks") / "mcdi502_s3_grupo1.ipynb"
carpeta_taller = Path.cwd()
for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / NOMBRE_NOTEBOOK).exists():
        carpeta_taller = base
        break
    if (base / "sumativa3" / "Taller II" / NOMBRE_NOTEBOOK).exists():
        carpeta_taller = base / "sumativa3" / "Taller II"
        break


def obtener_secreto(variable, archivo, clave, descripcion):
    """Devuelve el secreto sin imprimirlo: variable de entorno, archivo privado o entrada oculta."""
    valor = os.environ.get(variable)
    ruta = carpeta_taller / ".local" / archivo
    if not valor and ruta.exists():
        valor = json.loads(ruta.read_text(encoding="utf-8")).get(clave)
    if not valor:
        valor = getpass(f"{descripcion} (entrada oculta): ")
    if not valor:
        raise ValueError(f"Falta {descripcion}.")
    return valor.strip()


mongo_uri = obtener_secreto("MONGODB_URI", "mongodb.json", "uri", "URI de MongoDB entregada por el docente")
aqicn_token = obtener_secreto("AQICN_TOKEN", "aqicn.json", "token", "Token personal de AQICN")

CIUDAD = "Santiago"
LATITUD, LONGITUD = -33.4489, -70.6693        # Plaza de Armas de Santiago
ZONA_HORARIA = "America/Santiago"
REGISTROS = 100
fecha_extraccion_utc = datetime.now(timezone.utc)
print("Credenciales cargadas (no se muestran).")
print(f"Ciudad: {CIUDAD} ({LATITUD}, {LONGITUD}) · extracción UTC: {fecha_extraccion_utc:%Y-%m-%d %H:%M:%S}")

Credenciales cargadas (no se muestran).
Ciudad: Santiago (-33.4489, -70.6693) · extracción UTC: 2026-11-12 01:04:34


## 2. Ingesta de datos y creación de DataFrames (ID3.1)

Cada fuente produce un DataFrame de **100 registros** con las variables solicitadas, tipos explícitos, la fecha original de cada registro y una clave que identifica cada fila. La misma función de control verifica todas las fuentes: 100 filas, clave única y sin nulos, nulos por columna y cobertura temporal. Los nulos se informan, pero no se imputan.

In [4]:
def validar_fuente(nombre, df, claves, columna_fecha, filas=REGISTROS):
    """Controles de aceptación comunes a las cuatro fuentes."""
    cantidad = df.count()
    assert cantidad == filas, f"{nombre}: se requieren {filas} filas y hay {cantidad}"
    assert df.select(*claves).distinct().count() == cantidad, f"{nombre}: la clave {claves} se repite"
    assert df.filter(" OR ".join(f"{c} IS NULL" for c in claves)).count() == 0, f"{nombre}: clave nula"
    print(f"{nombre}: {cantidad} filas · clave única {claves}")
    df.printSchema()
    print("Nulos por columna:")
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)
    df.select(F.min(columna_fecha).alias("fecha_min"), F.max(columna_fecha).alias("fecha_max"),
              F.countDistinct(F.to_date(columna_fecha)).alias("dias_distintos")).show(truncate=False)

### 2.1 OpenMeteo → `df_openmeteo`

- **Consulta:** se usa el endpoint `forecast` de OpenMeteo, igual que el notebook base, pero en vez de repetir 20 veces la lectura actual (`current`) se pide la **serie horaria de los últimos 7 días completos** (`hourly`, `past_days=7`, `forecast_days=0`) para Santiago, en `America/Santiago`. Así cada fila corresponde a una hora distinta y real, y no a una copia de la misma lectura.
- **Variables:** temperatura a 2 m (°C), humedad relativa a 2 m (%), precipitación (mm), velocidad del viento a 10 m (km/h), dirección del viento a 10 m (°) e índice UV. Las unidades se validan con `hourly_units`.
- **Selección de 100 registros:** la serie tiene 168 horas. Se eligen 100 horas repartidas de forma uniforme y determinista en los siete días (`round(i·167/99)`, i = 0…99), que incluyen la primera y la última hora. La serie completa se conserva en `df_openmeteo_serie`, como respaldo para la consulta C3 de la última semana.

In [5]:
VARIABLES_OPENMETEO = {   # nombre en la API -> columna del DataFrame
    "temperature_2m": "temperatura_2m",
    "relative_humidity_2m": "humedad_relativa_2m",
    "precipitation": "precipitacion",
    "wind_speed_10m": "velocidad_viento_10m",
    "wind_direction_10m": "direccion_viento_10m",
    "uv_index": "indice_uv",
}
UNIDADES_ESPERADAS = {"temperature_2m": "°C", "relative_humidity_2m": "%", "precipitation": "mm",
                      "wind_speed_10m": "km/h", "wind_direction_10m": "°"}

try:
    respuesta = requests.get("https://api.open-meteo.com/v1/forecast", timeout=30, params={
        "latitude": LATITUD, "longitude": LONGITUD, "hourly": ",".join(VARIABLES_OPENMETEO),
        "past_days": 7, "forecast_days": 0, "timezone": ZONA_HORARIA})
    respuesta.raise_for_status()
    datos_openmeteo = respuesta.json()
except Exception as error:
    raise RuntimeError("No se pudo consultar OpenMeteo: " + type(error).__name__) from None

horario, unidades = datos_openmeteo["hourly"], datos_openmeteo["hourly_units"]
faltan = [v for v in VARIABLES_OPENMETEO if v not in horario]
assert not faltan, f"OpenMeteo no entregó: {faltan}"
for variable, unidad in UNIDADES_ESPERADAS.items():
    assert unidades[variable] == unidad, f"Unidad inesperada para {variable}: {unidades[variable]}"
horas = [datetime.fromisoformat(h) for h in horario["time"]]
assert len(horas) == 168 and len(set(horas)) == 168, f"Se esperaban 168 horas distintas; hay {len(horas)}"

filas_openmeteo = [
    (i, hora, CIUDAD, *[None if horario[v][i] is None else float(horario[v][i]) for v in VARIABLES_OPENMETEO])
    for i, hora in enumerate(horas)
]
schema_openmeteo = T.StructType([
    T.StructField("indice_hora", T.IntegerType(), False),
    T.StructField("fecha_hora", T.TimestampNTZType(), False),   # hora local de Santiago, como la entrega la API
    T.StructField("ciudad", T.StringType(), False),
    *[T.StructField(c, T.DoubleType(), True) for c in VARIABLES_OPENMETEO.values()],
])
df_openmeteo_serie = spark.createDataFrame(filas_openmeteo, schema_openmeteo)

ultimo = len(horas) - 1
indices_muestra = sorted({round(i * ultimo / (REGISTROS - 1)) for i in range(REGISTROS)})
assert len(indices_muestra) == REGISTROS
df_openmeteo = (df_openmeteo_serie.filter(F.col("indice_hora").isin(indices_muestra))
                .orderBy("fecha_hora"))

print(f"Ubicación devuelta: {datos_openmeteo['latitude']}, {datos_openmeteo['longitude']} · zona {datos_openmeteo['timezone']}")
print(f"Serie horaria: {len(horas)} horas, del {horas[0]} al {horas[-1]} (corte de la serie)")
print("Unidades:", {VARIABLES_OPENMETEO[k]: unidades[k] for k in VARIABLES_OPENMETEO})

Ubicación devuelta: -33.427067, -70.64276 · zona America/Santiago
Serie horaria: 168 horas, del 2026-11-04 00:00:00 al 2026-11-10 23:00:00 (corte de la serie)
Unidades: {'temperatura_2m': '°C', 'humedad_relativa_2m': '%', 'precipitacion': 'mm', 'velocidad_viento_10m': 'km/h', 'direccion_viento_10m': '°', 'indice_uv': ''}


In [6]:
validar_fuente("df_openmeteo", df_openmeteo, ["fecha_hora"], "fecha_hora")
df_openmeteo.show(10, truncate=False)

df_openmeteo: 100 filas · clave única ['fecha_hora']
root
 |-- indice_hora: integer (nullable = false)
 |-- fecha_hora: timestamp_ntz (nullable = false)
 |-- ciudad: string (nullable = false)
 |-- temperatura_2m: double (nullable = true)
 |-- humedad_relativa_2m: double (nullable = true)
 |-- precipitacion: double (nullable = true)
 |-- velocidad_viento_10m: double (nullable = true)
 |-- direccion_viento_10m: double (nullable = true)
 |-- indice_uv: double (nullable = true)

Nulos por columna:


+-----------+----------+------+--------------+-------------------+-------------+--------------------+--------------------+---------+
|indice_hora|fecha_hora|ciudad|temperatura_2m|humedad_relativa_2m|precipitacion|velocidad_viento_10m|direccion_viento_10m|indice_uv|
+-----------+----------+------+--------------+-------------------+-------------+--------------------+--------------------+---------+
|0          |0         |0     |0             |0                  |0            |0                   |0                   |0        |
+-----------+----------+------+--------------+-------------------+-------------+--------------------+--------------------+---------+



+-------------------+-------------------+--------------+
|fecha_min          |fecha_max          |dias_distintos|
+-------------------+-------------------+--------------+
|2026-11-04 00:00:00|2026-11-10 23:00:00|7             |
+-------------------+-------------------+--------------+



+-----------+-------------------+--------+--------------+-------------------+-------------+--------------------+--------------------+---------+
|indice_hora|fecha_hora         |ciudad  |temperatura_2m|humedad_relativa_2m|precipitacion|velocidad_viento_10m|direccion_viento_10m|indice_uv|
+-----------+-------------------+--------+--------------+-------------------+-------------+--------------------+--------------------+---------+
|0          |2026-11-04 00:00:00|Santiago|12.7          |88.0               |0.0          |0.4                 |180.0               |0.0      |
|2          |2026-11-04 02:00:00|Santiago|12.3          |88.0               |0.0          |0.8                 |333.0               |0.0      |
|3          |2026-11-04 03:00:00|Santiago|12.1          |92.0               |0.0          |1.0                 |239.0               |0.0      |
|5          |2026-11-04 05:00:00|Santiago|11.1          |92.0               |0.0          |2.2                 |336.0               |0.0

### 2.2 AQICN → `df_aqicn`

- **Token:** el notebook base usa el token `demo`, pero ese token devuelve siempre los datos de **Shanghái**, aunque se pida otra ciudad. Por eso se usa un token personal (gratuito) y se rechaza cualquier respuesta cuyas coordenadas no estén en Chile.
- **Por qué no repetir la consulta:** el feed de una ciudad entrega solo la última lectura horaria de una estación. Repetirlo 100 veces, como hace el ejemplo, produce 100 copias del mismo valor.
- **Estrategia:** se buscan las estaciones de la región con `map/bounds`, comenzando por la Región Metropolitana. Si no alcanzan, la búsqueda se amplía a la zona central y luego a Chile. Las estaciones se ordenan por distancia a Santiago (desempate por `uid`) y se consulta el feed de cada una (`feed/@uid`). La primera ronda entrega una observación real por estación chilena con contaminantes.
- **Variables:** PM2.5, PM10, O3, NO2 y SO2 desde `iaqi`. AQICN entrega cada contaminante como **sub-índice AQI (escala US EPA)**, no como concentración en µg/m³. Las consultas se interpretan en esa escala. Algunas estaciones no miden todos los contaminantes: esos valores quedan nulos y se informan.
- **Completar las 100 filas:** AQICN publica solo unas decenas de estaciones para todo Chile, menos de las 100 requeridas (la cantidad exacta se imprime abajo). Por eso, después de la primera ronda se repite la consulta a las mismas estaciones, en el mismo orden, hasta completar 100 capturas, como hace el notebook base. Cada captura repetida queda identificada en la columna `ronda` y conserva su `fecha_captura`; la clave de la fila es (`estacion_uid`, `ronda`). No se inventan ni se imputan valores.
- **Vigencia:** cada fila guarda `fecha_observacion`, la hora de la lectura según la estación. Si la red de monitoreo no ha publicado datos recientes, las rondas repiten la misma lectura. La celda de control informa la antigüedad de las observaciones para que esto quede a la vista.

In [7]:
CONTAMINANTES = {"pm25": "pm25", "pm10": "pm10", "o3": "o3", "no2": "no2", "so2": "so2"}
CAJAS_BUSQUEDA = [   # (nombre, (lat_sur, lon_oeste, lat_norte, lon_este)), de menor a mayor
    ("Región Metropolitana", (-34.30, -71.75, -32.90, -69.75)),
    ("Zona central", (-36.60, -72.90, -32.00, -69.75)),
    ("Chile", (-56.00, -76.00, -17.50, -66.00)),
]


SESION_AQICN = requests.Session()   # reutiliza la conexión HTTPS: ~3 veces más rápido que abrir una por consulta


def consultar_aqicn(ruta, **parametros):
    """GET a AQICN sin exponer el token en errores ni salidas."""
    try:
        respuesta = SESION_AQICN.get(f"https://api.waqi.info{ruta}", timeout=30,
                                 params={**parametros, "token": aqicn_token})
        respuesta.raise_for_status()
        datos = respuesta.json()
    except Exception as error:
        raise RuntimeError(f"No se pudo consultar AQICN {ruta.split('@')[0]}: {type(error).__name__}") from None
    if datos.get("status") != "ok":
        detalle = datos.get("data") if isinstance(datos.get("data"), str) else "sin detalle"
        raise RuntimeError(f"AQICN respondió '{datos.get('status')}': {detalle}")
    return datos["data"]


def distancia_km(lat, lon):
    """Distancia de Haversine a Santiago."""
    p1, p2 = math.radians(LATITUD), math.radians(lat)
    dp, dl = p2 - p1, math.radians(lon - LONGITUD)
    return 6371 * 2 * math.asin(math.sqrt(math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2))


def es_de_chile(feed):
    ciudad = feed.get("city", {})
    lat = (ciudad.get("geo") or [0, 0])[0]
    texto = f"{ciudad.get('url', '')} {ciudad.get('name', '')}".lower()
    return -56 <= lat <= -17 and "chile" in texto


# 1) Estaciones candidatas, ampliando la búsqueda solo si hace falta
candidatas, region_usada = {}, None
for region, caja in CAJAS_BUSQUEDA:
    for estacion in consultar_aqicn("/map/bounds/", latlng=",".join(map(str, caja))):
        candidatas[estacion["uid"]] = estacion      # aqi "-" = sin índice vigente; el feed puede traer contaminantes
    region_usada = region
    vigentes = sum(str(e.get("aqi", "")).strip().isdigit() for e in candidatas.values())
    print(f"{region}: {len(candidatas)} estaciones ({vigentes} con AQI vigente)")
    if len(candidatas) >= REGISTROS * 1.3:                          # margen para descartes
        break
orden = sorted(candidatas.values(), key=lambda e: (distancia_km(e["lat"], e["lon"]), e["uid"]))

# 2) Feed de cada estación, de la más cercana a la más lejana
filas_aqicn, descartadas = [], {"fuera de Chile": 0, "sin contaminantes": 0, "error": 0}
for estacion in orden:
    if len(filas_aqicn) == REGISTROS:
        break
    try:
        feed = consultar_aqicn(f"/feed/@{estacion['uid']}/")
    except RuntimeError:
        descartadas["error"] += 1
        continue
    if not es_de_chile(feed):
        descartadas["fuera de Chile"] += 1
        continue
    iaqi = feed.get("iaqi", {})
    valores = [None if iaqi.get(c, {}).get("v") is None else float(iaqi[c]["v"]) for c in CONTAMINANTES]
    if all(v is None for v in valores):
        descartadas["sin contaminantes"] += 1
        continue
    lat, lon = feed["city"]["geo"][:2]
    filas_aqicn.append((int(estacion["uid"]), feed["city"]["name"], float(lat), float(lon),
                        round(distancia_km(lat, lon), 1), datetime.fromisoformat(feed["time"]["iso"]),
                        datetime.now(timezone.utc), 1, *valores))
    time.sleep(0.2)                                                   # cortesía con la API

# 3) Respaldo documentado: rondas adicionales solo si no hubo 100 estaciones
estaciones_distintas = len(filas_aqicn)
ronda = 1
while 0 < len(filas_aqicn) < REGISTROS:
    ronda += 1
    for fila in filas_aqicn[:estaciones_distintas]:
        if len(filas_aqicn) == REGISTROS:
            break
        feed = consultar_aqicn(f"/feed/@{fila[0]}/")
        iaqi = feed.get("iaqi", {})
        valores = [None if iaqi.get(c, {}).get("v") is None else float(iaqi[c]["v"]) for c in CONTAMINANTES]
        filas_aqicn.append((*fila[:5], datetime.fromisoformat(feed["time"]["iso"]),
                            datetime.now(timezone.utc), ronda, *valores))
        time.sleep(0.5)

schema_aqicn = T.StructType([
    T.StructField("estacion_uid", T.IntegerType(), False),
    T.StructField("estacion", T.StringType(), False),
    T.StructField("latitud", T.DoubleType(), False),
    T.StructField("longitud", T.DoubleType(), False),
    T.StructField("distancia_km", T.DoubleType(), False),
    T.StructField("fecha_observacion", T.TimestampType(), False),   # instante con zona, mostrado en America/Santiago
    T.StructField("fecha_captura", T.TimestampType(), False),
    T.StructField("ronda", T.IntegerType(), False),
    *[T.StructField(c, T.DoubleType(), True) for c in CONTAMINANTES.values()],
])
df_aqicn = spark.createDataFrame(filas_aqicn, schema_aqicn).orderBy("distancia_km", "estacion_uid", "ronda")

print(f"Búsqueda hasta: {region_usada} · estaciones distintas: {estaciones_distintas} · rondas: {ronda}")
print("Estaciones descartadas:", descartadas)

Región Metropolitana: 15 estaciones (0 con AQI vigente)


Zona central: 29 estaciones (0 con AQI vigente)


Chile: 50 estaciones (0 con AQI vigente)


Búsqueda hasta: Chile · estaciones distintas: 50 · rondas: 2
Estaciones descartadas: {'fuera de Chile': 0, 'sin contaminantes': 0, 'error': 0}


In [8]:
validar_fuente("df_aqicn", df_aqicn, ["estacion_uid", "ronda"], "fecha_observacion")
print("Estaciones distintas:", df_aqicn.select("estacion_uid").distinct().count(),
      "· observaciones distintas (estación, hora de observación):",
      df_aqicn.select("estacion_uid", "fecha_observacion").distinct().count())
print("Capturas por ronda:")
df_aqicn.groupBy("ronda").count().orderBy("ronda").show()
print("Antigüedad de las observaciones al momento de la captura (horas):")
df_aqicn.select(((F.unix_timestamp("fecha_captura") - F.unix_timestamp("fecha_observacion")) / 3600)
                .alias("horas")).agg(F.round(F.min("horas"), 1).alias("minima"),
                                     F.round(F.max("horas"), 1).alias("maxima")).show()
print("Valores válidos por contaminante (escala AQI US EPA):")
df_aqicn.select([F.count(c).alias(c) for c in CONTAMINANTES.values()]).show()
df_aqicn.select("estacion_uid", "estacion", "distancia_km", "fecha_observacion",
                *CONTAMINANTES.values()).show(10, truncate=40)

df_aqicn: 100 filas · clave única ['estacion_uid', 'ronda']
root
 |-- estacion_uid: integer (nullable = false)
 |-- estacion: string (nullable = false)
 |-- latitud: double (nullable = false)
 |-- longitud: double (nullable = false)
 |-- distancia_km: double (nullable = false)
 |-- fecha_observacion: timestamp (nullable = false)
 |-- fecha_captura: timestamp (nullable = false)
 |-- ronda: integer (nullable = false)
 |-- pm25: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- o3: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- so2: double (nullable = true)

Nulos por columna:


+------------+--------+-------+--------+------------+-----------------+-------------+-----+----+----+---+---+---+
|estacion_uid|estacion|latitud|longitud|distancia_km|fecha_observacion|fecha_captura|ronda|pm25|pm10|o3 |no2|so2|
+------------+--------+-------+--------+------------+-----------------+-------------+-----+----+----+---+---+---+
|0           |0       |0      |0       |0           |0                |0            |0    |2   |26  |68 |66 |74 |
+------------+--------+-------+--------+------------+-----------------+-------------+-----+----+----+---+---+---+



+-------------------+-------------------+--------------+
|fecha_min          |fecha_max          |dias_distintos|
+-------------------+-------------------+--------------+
|2026-11-09 13:00:00|2026-11-09 21:00:00|1             |
+-------------------+-------------------+--------------+



Estaciones distintas: 50 · observaciones distintas (estación, hora de observación): 50
Capturas por ronda:


+-----+-----+
|ronda|count|
+-----+-----+
|    1|   50|
|    2|   50|
+-----+-----+

Antigüedad de las observaciones al momento de la captura (horas):


+------+------+
|minima|maxima|
+------+------+
|  49.1|  57.1|
+------+------+

Valores válidos por contaminante (escala AQI US EPA):


+----+----+---+---+---+
|pm25|pm10| o3|no2|so2|
+----+----+---+---+---+
|  98|  74| 32| 34| 26|
+----+----+---+---+---+



+------------+-----------------------+------------+-------------------+----+----+----+----+----+
|estacion_uid|               estacion|distancia_km|  fecha_observacion|pm25|pm10|  o3| no2| so2|
+------------+-----------------------+------------+-------------------+----+----+----+----+----+
|         431|Parque O'Higgins, Chile|         1.9|2026-11-09 20:00:00|25.0|29.0|18.4|12.7|NULL|
|         431|Parque O'Higgins, Chile|         1.9|2026-11-09 20:00:00|25.0|29.0|18.4|12.7|NULL|
|         435|     Cerro Navia, Chile|         6.1|2026-11-09 21:00:00|50.0|26.0| 8.8|11.7|NULL|
|         435|     Cerro Navia, Chile|         6.1|2026-11-09 21:00:00|50.0|26.0| 8.8|11.7|NULL|
|       14731|    Cerrillos II, Chile|         6.1|2026-11-09 20:00:00|25.0|38.0|NULL|NULL|NULL|
|       14731|    Cerrillos II, Chile|         6.1|2026-11-09 20:00:00|25.0|38.0|NULL|NULL|NULL|
|         432|        Pudahuel, Chile|         7.6|2026-11-09 20:00:00|42.0|37.0|10.4|14.9|NULL|
|         432|        Pudahuel

### 2.3 MongoDB (ESP32) → `df_mongo_bpm`

*Bloque de Sebastián, adaptado de `-BaseEjemploMongoDB.ipynb`.*

- **Origen:** colección `telemetria.lecturas` del ejemplo del docente, leída con PyMongo en modo lectura.
- **Selección:** las 100 lecturas más recientes, ordenadas por `fecha` descendente y `_id` descendente. El orden se aplica en el servidor antes de limitar.
- **Mapeo de campos:** `altitudFt` → `alturaft`, `presionAtm` → `presion_atmosferica`, `ppmCO` → `monoxido_carbono`. También se conserva `presionPa` como `presion_pa`. Los nombres sugieren atm, ppm y Pa como unidades.
- **Fechas:** las fechas BSON son instantes UTC, y Spark las muestra en `America/Santiago`.
- **Tipado:** los valores se reciben como texto y se convierten con `try_cast`. El conteo de valores no convertibles se informa.

In [9]:
from pymongo import MongoClient


def texto_o_nulo(valor):
    return None if valor is None else str(valor)

globals().pop('df_mongo_bpm', None)
try:
    with MongoClient(mongo_uri, serverSelectionTimeoutMS=15000,
                     connectTimeoutMS=15000, socketTimeoutMS=30000,
                     tz_aware=True) as cliente:
        documentos = list(cliente['telemetria']['lecturas'].find(
            {}, {'altitudFt': 1, 'presionAtm': 1, 'presionPa': 1,
                 'ppmCO': 1, 'fecha': 1}
        ).sort([('fecha', -1), ('_id', -1)]).limit(100))
except Exception as error:
    # No exponer la URI ni el mensaje del servidor en las salidas.
    raise RuntimeError('No se pudo leer MongoDB: ' + type(error).__name__ +
                       '. Revisar acceso, red y URI.') from None

if len(documentos) != 100:
    raise ValueError(f'MongoDB devolvió {len(documentos)} registros; se requieren 100.')
campos_mongo = {'_id', 'fecha', 'altitudFt', 'presionAtm', 'ppmCO'}
if not campos_mongo.issubset(set().union(*(d.keys() for d in documentos))):
    raise ValueError('La colección no contiene todos los campos solicitados.')
if any(not isinstance(d.get('fecha'), datetime) for d in documentos):
    raise ValueError('Revisar fechas BSON faltantes o no válidas antes de seleccionar filas.')

filas_mongo = [(
    str(d['_id']), d['fecha'], texto_o_nulo(d.get('altitudFt')),
    texto_o_nulo(d.get('presionAtm')), texto_o_nulo(d.get('presionPa')),
    texto_o_nulo(d.get('ppmCO'))
) for d in documentos]
schema_mongo = T.StructType([
    T.StructField('id', T.StringType(), False),
    T.StructField('fecha', T.TimestampType(), False),
    *[T.StructField(c, T.StringType(), True) for c in
      ['alturaft', 'presion_atmosferica', 'presion_pa', 'monoxido_carbono']]
])
mongo_original = spark.createDataFrame(filas_mongo, schema_mongo)
df_mongo_bpm = mongo_original
for campo in ['alturaft', 'presion_atmosferica', 'presion_pa', 'monoxido_carbono']:
    fallos = df_mongo_bpm.filter(
        F.col(campo).isNotNull() & F.expr(f'try_cast({campo} AS DOUBLE)').isNull()
    ).count()
    print(f'MongoDB {campo}: valores no convertibles = {fallos}')
    df_mongo_bpm = df_mongo_bpm.withColumn(campo, F.expr(f'try_cast({campo} AS DOUBLE)'))
df_mongo_bpm = df_mongo_bpm.orderBy(F.col('fecha').desc(), F.col('id').desc())
# altura_metros se calcula en la sección 3.1

MongoDB alturaft: valores no convertibles = 0


MongoDB presion_atmosferica: valores no convertibles = 0


MongoDB presion_pa: valores no convertibles = 0


MongoDB monoxido_carbono: valores no convertibles = 0


### 2.4 MySQL (ESP32) → `df_mysql_dht`

*Bloque de Sebastián, adaptado de `-BaseEjemploMysqlPHP_APIRest.ipynb`.*

- **Origen:** endpoint PHP del curso (`obtener_datos.php`), que expone la tabla MySQL como JSON.
- **Variables:** temperatura (°C), que se renombra `temperatura_interior` según el enunciado, y humedad (%).
- **Fechas:** se conservan `fecha_original` (el texto recibido) y `fecha` (`TIMESTAMP_NTZ`), porque la fuente no declara zona horaria.
- **Limpieza:** se eliminan los duplicados exactos y se rechazan los IDs con registros contradictorios. Luego se toman los 100 más recientes por `fecha` e `id`.

In [10]:
MYSQL_URL = 'http://mgtt.comunidadingenieria.cl/obtener_datos.php'

try:
    respuesta = requests.get(MYSQL_URL, timeout=30)
    respuesta.raise_for_status()
    datos_mysql = respuesta.json()
except Exception as error:
    raise RuntimeError('No se pudo leer MySQL/REST: ' + type(error).__name__) from None
if not isinstance(datos_mysql, list) or not datos_mysql:
    raise ValueError('Se esperaba una lista JSON no vacía de registros.')
campos_mysql = {'id', 'temperatura', 'humedad', 'fecha'}
if any(not isinstance(d, dict) or not campos_mysql.issubset(d) for d in datos_mysql):
    raise ValueError('El JSON no coincide con el esquema del notebook base.')

filas_mysql = [tuple(texto_o_nulo(d[c]) for c in ['id','temperatura','humedad','fecha'])
               for d in datos_mysql]
mysql_original = spark.createDataFrame(
    filas_mysql, 'id_original STRING, temperatura_interior STRING, humedad STRING, fecha_original STRING')
df_mysql_tipado = mysql_original
for campo, tipo in [('id_original', 'BIGINT'), ('temperatura_interior', 'DOUBLE'),
                    ('humedad', 'DOUBLE'), ('fecha_original', 'TIMESTAMP_NTZ')]:
    conversion = F.expr(f'try_cast({campo} AS {tipo})')
    fallos = df_mysql_tipado.filter(F.col(campo).isNotNull() & conversion.isNull()).count()
    print(f'MySQL {campo}: valores no convertibles = {fallos}')
    destino = {'id_original': 'id', 'fecha_original': 'fecha'}.get(campo, campo)
    df_mysql_tipado = df_mysql_tipado.withColumn(destino, conversion)
if df_mysql_tipado.filter(F.col('id').isNull() | F.col('fecha').isNull()).count():
    raise ValueError('Hay IDs o fechas no válidos; corregir antes de ordenar y seleccionar.')
sin_duplicados = df_mysql_tipado.dropDuplicates()
if sin_duplicados.groupBy('id').count().filter(F.col('count') > 1).count():
    raise ValueError('Hay IDs con registros contradictorios; revisar la fuente.')
print('Filas recibidas:', len(datos_mysql))
print('Duplicados exactos eliminados:', df_mysql_tipado.count() - sin_duplicados.count())
df_mysql_dht = (sin_duplicados.orderBy(F.col('fecha').desc(), F.col('id').desc())
                .limit(100).drop('id_original'))

MySQL id_original: valores no convertibles = 0


MySQL temperatura_interior: valores no convertibles = 0


MySQL humedad: valores no convertibles = 0


MySQL fecha_original: valores no convertibles = 0


Filas recibidas: 1000


Duplicados exactos eliminados: 0


In [11]:
validar_fuente("df_mongo_bpm", df_mongo_bpm, ["id"], "fecha")
validar_fuente("df_mysql_dht", df_mysql_dht, ["id"], "fecha")
df_mysql_dht.show(10, truncate=False)

df_mongo_bpm: 100 filas · clave única ['id']
root
 |-- id: string (nullable = false)
 |-- fecha: timestamp (nullable = false)
 |-- alturaft: double (nullable = true)
 |-- presion_atmosferica: double (nullable = true)
 |-- presion_pa: double (nullable = true)
 |-- monoxido_carbono: double (nullable = true)

Nulos por columna:


+---+-----+--------+-------------------+----------+----------------+
|id |fecha|alturaft|presion_atmosferica|presion_pa|monoxido_carbono|
+---+-----+--------+-------------------+----------+----------------+
|0  |0    |0       |0                  |0         |0               |
+---+-----+--------+-------------------+----------+----------------+



+-----------------------+-----------------------+--------------+
|fecha_min              |fecha_max              |dias_distintos|
+-----------------------+-----------------------+--------------+
|2026-11-11 13:13:27.325|2026-11-11 13:41:18.504|1             |
+-----------------------+-----------------------+--------------+



df_mysql_dht: 100 filas · clave única ['id']
root
 |-- temperatura_interior: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- fecha_original: string (nullable = true)
 |-- id: long (nullable = true)
 |-- fecha: timestamp_ntz (nullable = true)

Nulos por columna:


+--------------------+-------+--------------+---+-----+
|temperatura_interior|humedad|fecha_original|id |fecha|
+--------------------+-------+--------------+---+-----+
|0                   |0      |0             |0  |0    |
+--------------------+-------+--------------+---+-----+



+-------------------+-------------------+--------------+
|fecha_min          |fecha_max          |dias_distintos|
+-------------------+-------------------+--------------+
|2026-11-11 08:13:27|2026-11-11 08:41:18|1             |
+-------------------+-------------------+--------------+



+--------------------+-------+-------------------+------+-------------------+
|temperatura_interior|humedad|fecha_original     |id    |fecha              |
+--------------------+-------+-------------------+------+-------------------+
|24.1                |62.6   |2026-11-11 08:41:18|370874|2026-11-11 08:41:18|
|24.1                |62.8   |2026-11-11 08:41:01|370873|2026-11-11 08:41:01|
|24.1                |62.7   |2026-11-11 08:40:44|370872|2026-11-11 08:40:44|
|24.1                |62.5   |2026-11-11 08:40:27|370871|2026-11-11 08:40:27|
|24.1                |62.4   |2026-11-11 08:40:10|370870|2026-11-11 08:40:10|
|24.1                |62.4   |2026-11-11 08:39:53|370869|2026-11-11 08:39:53|
|24.1                |62.4   |2026-11-11 08:39:37|370868|2026-11-11 08:39:37|
|24.2                |62.3   |2026-11-11 08:39:20|370867|2026-11-11 08:39:20|
|24.1                |62.3   |2026-11-11 08:39:03|370866|2026-11-11 08:39:03|
|24.1                |62.3   |2026-11-11 08:38:46|370865|2026-11

## 3. Preprocesamiento y transformación de datos (RA4, ID4.2)

### 3.1 `altura_metros` en `df_mongo_bpm`

*Bloque de Sebastián.* `alturaft` se convierte a número y se multiplica por 0,3048 (1 pie = 0,3048 m). Los valores faltantes o no convertibles quedan como nulos y no se imputan. Antes de aplicarla a los datos reales, la función se prueba con casos aislados: 10 pies, un valor faltante y un texto no numérico.

In [12]:
def agregar_altura_metros(df):
    return (df.withColumn('alturaft', F.expr('try_cast(alturaft AS DOUBLE)'))
              .withColumn('altura_metros', F.col('alturaft') * F.lit(0.3048)))

# Casos de prueba aislados; no se utilizan para completar las 100 filas reales.
prueba_altura = spark.createDataFrame(
    [('conocido', '10'), ('faltante', None), ('invalido', 'sin dato')],
    'caso STRING, alturaft STRING'
)
resultado_prueba = agregar_altura_metros(prueba_altura)
valores = {fila.caso: fila.altura_metros for fila in resultado_prueba.collect()}
assert abs(valores['conocido'] - 3.048) < 1e-9
assert valores['faltante'] is None
assert valores['invalido'] is None
resultado_prueba.show()
print('Conversión de altura verificada')


df_mongo_bpm = agregar_altura_metros(df_mongo_bpm).orderBy(F.col("fecha").desc(), F.col("id").desc())
assert df_mongo_bpm.count() == REGISTROS
assert df_mongo_bpm.filter(
    ~F.col("altura_metros").eqNullSafe(F.col("alturaft") * F.lit(0.3048))).count() == 0
print("altura_metros aplicada a las", REGISTROS, "filas de df_mongo_bpm")
df_mongo_bpm.select("id", "fecha", "alturaft", "altura_metros").show(5, truncate=False)

+--------+--------+-------------+
|    caso|alturaft|altura_metros|
+--------+--------+-------------+
|conocido|    10.0|        3.048|
|faltante|    NULL|         NULL|
|invalido|    NULL|         NULL|
+--------+--------+-------------+

Conversión de altura verificada


altura_metros aplicada a las 100 filas de df_mongo_bpm


+------------------------+-----------------------+--------+------------------+
|id                      |fecha                  |alturaft|altura_metros     |
+------------------------+-----------------------+--------+------------------+
|6af49aaec1ab6b5957d4b188|2026-11-11 13:41:18.504|827.78  |252.307344        |
|6af49a9dc1ab6b5957d4b186|2026-11-11 13:41:01.683|826.1   |251.79528000000002|
|6af49a8cc1ab6b5957d4b184|2026-11-11 13:40:44.9  |826.66  |251.965968        |
|6af49a7bc1ab6b5957d4b182|2026-11-11 13:40:27.988|826.66  |251.965968        |
|6af49a6bc1ab6b5957d4b180|2026-11-11 13:40:11.018|826.1   |251.79528000000002|
+------------------------+-----------------------+--------+------------------+
only showing top 5 rows


### 3.2 `condicion_clima` en `df_openmeteo`

Las reglas se evalúan **en el orden del enunciado** con `when` encadenados, y gana la primera que se cumple:

| Regla | Resultado | Caso de prueba (frontera) |
|---|---|---|
| Temperatura > 30 °C y humedad > 70 % | CALOR Y HUMEDAD | 31 °C y 71 % |
| Temperatura > 30 °C y humedad ≤ 70 % | CALOR SECO | 31 °C y 70 % |
| Temperatura ≤ 30 °C | CLIMA MODERADO | 30 °C, incluso con humedad nula |
| Cualquier otro caso | CONDICION DESCONOCIDA | 31 °C con humedad nula; temperatura nula |

En Spark, una comparación con un valor nulo no es verdadera. Por eso, un registro con temperatura nula, o con más de 30 °C y humedad nula, no cumple ninguna de las tres primeras reglas y queda como CONDICION DESCONOCIDA en vez de clasificarse mal.

In [13]:
def agregar_condicion_clima(df, temperatura="temperatura_2m", humedad="humedad_relativa_2m"):
    t, h = F.col(temperatura), F.col(humedad)
    return df.withColumn(
        "condicion_clima",
        F.when((t > 30) & (h > 70), "CALOR Y HUMEDAD")
         .when((t > 30) & (h <= 70), "CALOR SECO")
         .when(t <= 30, "CLIMA MODERADO")
         .otherwise("CONDICION DESCONOCIDA"))


# Casos de prueba aislados; no se agregan a las 100 filas reales.
casos = [(31.0, 71.0, "CALOR Y HUMEDAD"), (31.0, 70.0, "CALOR SECO"), (30.0, None, "CLIMA MODERADO"),
         (12.5, 95.0, "CLIMA MODERADO"), (31.0, None, "CONDICION DESCONOCIDA"),
         (None, 50.0, "CONDICION DESCONOCIDA")]
prueba_clima = agregar_condicion_clima(spark.createDataFrame(
    casos, "temperatura_2m DOUBLE, humedad_relativa_2m DOUBLE, esperado STRING"))
prueba_clima.show(truncate=False)
assert prueba_clima.filter(F.col("condicion_clima") != F.col("esperado")).count() == 0
print("Las cuatro ramas de condicion_clima pasaron sus pruebas")

df_openmeteo = agregar_condicion_clima(df_openmeteo).orderBy("fecha_hora")
assert df_openmeteo.count() == REGISTROS
print("Distribución en las", REGISTROS, "horas de df_openmeteo:")
df_openmeteo.groupBy("condicion_clima").agg(
    F.count("*").alias("horas"), F.round(F.max("temperatura_2m"), 1).alias("temp_max")
).orderBy(F.desc("horas")).show(truncate=False)
df_openmeteo.select("fecha_hora", "temperatura_2m", "humedad_relativa_2m", "condicion_clima").show(10, truncate=False)

+--------------+-------------------+---------------------+---------------------+
|temperatura_2m|humedad_relativa_2m|esperado             |condicion_clima      |
+--------------+-------------------+---------------------+---------------------+
|31.0          |71.0               |CALOR Y HUMEDAD      |CALOR Y HUMEDAD      |
|31.0          |70.0               |CALOR SECO           |CALOR SECO           |
|30.0          |NULL               |CLIMA MODERADO       |CLIMA MODERADO       |
|12.5          |95.0               |CLIMA MODERADO       |CLIMA MODERADO       |
|31.0          |NULL               |CONDICION DESCONOCIDA|CONDICION DESCONOCIDA|
|NULL          |50.0               |CONDICION DESCONOCIDA|CONDICION DESCONOCIDA|
+--------------+-------------------+---------------------+---------------------+



Las cuatro ramas de condicion_clima pasaron sus pruebas


Distribución en las 100 horas de df_openmeteo:


+---------------+-----+--------+
|condicion_clima|horas|temp_max|
+---------------+-----+--------+
|CLIMA MODERADO |97   |28.0    |
|CALOR SECO     |3    |31.7    |
+---------------+-----+--------+



+-------------------+--------------+-------------------+---------------+
|fecha_hora         |temperatura_2m|humedad_relativa_2m|condicion_clima|
+-------------------+--------------+-------------------+---------------+
|2026-11-04 00:00:00|12.7          |88.0               |CLIMA MODERADO |
|2026-11-04 02:00:00|12.3          |88.0               |CLIMA MODERADO |
|2026-11-04 03:00:00|12.1          |92.0               |CLIMA MODERADO |
|2026-11-04 05:00:00|11.1          |92.0               |CLIMA MODERADO |
|2026-11-04 07:00:00|10.5          |95.0               |CLIMA MODERADO |
|2026-11-04 08:00:00|11.7          |94.0               |CLIMA MODERADO |
|2026-11-04 10:00:00|14.8          |78.0               |CLIMA MODERADO |
|2026-11-04 12:00:00|18.3          |62.0               |CLIMA MODERADO |
|2026-11-04 13:00:00|19.7          |56.0               |CLIMA MODERADO |
|2026-11-04 15:00:00|21.9          |46.0               |CLIMA MODERADO |
+-------------------+--------------+---------------

### Control de traspaso

Esta tabla resume las cuatro fuentes ya transformadas, como punto de partida para la integración. Cada fila indica la clave, el orden documentado y la cobertura temporal de un DataFrame.

In [14]:
fuentes = {
    "df_openmeteo": (df_openmeteo, "fecha_hora", "fecha_hora asc", "hora local Santiago (API)"),
    "df_aqicn": (df_aqicn, "fecha_observacion", "distancia_km, estacion_uid, ronda", "instante con zona"),
    "df_mongo_bpm": (df_mongo_bpm, "fecha", "fecha desc, id desc", "instante UTC (BSON)"),
    "df_mysql_dht": (df_mysql_dht, "fecha", "fecha desc, id desc", "sin zona declarada"),
}
resumen = []
for nombre, (df, col_fecha, orden_doc, zona) in fuentes.items():
    extremos = df.agg(F.min(col_fecha).cast("string"), F.max(col_fecha).cast("string")).first()
    resumen.append((nombre, df.count(), len(df.columns), orden_doc, zona, extremos[0][:19], extremos[1][:19]))
    assert df.count() == REGISTROS
assert "altura_metros" in df_mongo_bpm.columns and "condicion_clima" in df_openmeteo.columns
spark.createDataFrame(resumen, "dataframe STRING, filas INT, columnas INT, orden STRING, zona_fecha STRING, "
                               "desde STRING, hasta STRING").show(truncate=False)
print("Cuatro DataFrames de 100 filas con altura_metros y condicion_clima, listos para la integración.")

+------------+-----+--------+---------------------------------+-------------------------+-------------------+-------------------+
|dataframe   |filas|columnas|orden                            |zona_fecha               |desde              |hasta              |
+------------+-----+--------+---------------------------------+-------------------------+-------------------+-------------------+
|df_openmeteo|100  |10      |fecha_hora asc                   |hora local Santiago (API)|2026-11-04 00:00:00|2026-11-10 23:00:00|
|df_aqicn    |100  |13      |distancia_km, estacion_uid, ronda|instante con zona        |2026-11-09 13:00:00|2026-11-09 21:00:00|
|df_mongo_bpm|100  |7       |fecha desc, id desc              |instante UTC (BSON)      |2026-11-11 13:13:27|2026-11-11 13:41:18|
|df_mysql_dht|100  |5       |fecha desc, id desc              |sin zona declarada       |2026-11-11 08:13:27|2026-11-11 08:41:18|
+------------+-----+--------+---------------------------------+-------------------------+-

## 4. Integración de datos (RA3)

*Pendiente: bloque de Yerko.*

Información para definir la estrategia de integración:

- Las fuentes **no comparten un período común**. OpenMeteo cubre 7 días pasados, AQICN la hora actual de 100 estaciones, MongoDB unos 30 minutos recientes y MySQL otros 30 minutos de una zona horaria no declarada (cerca de 5 horas de desfase con MongoDB, según el bloque de Sebastián). Un join por fecha dejaría a casi todas las filas sin pareja.
- **Recomendación:** usar un índice ordinal reproducible por fuente (`row_number()` sobre el orden documentado en el control de traspaso, con desempate estable) y unir las cuatro fuentes por ese índice. Hay que conservar las fechas y claves originales de cada una y documentar que el emparejamiento solo presenta los datos juntos, sin suponer que ocurrieron en el mismo momento.
- Para no repetir nombres de columna, se recomienda usar prefijos por fuente (`om_`, `aq_`, `mg_`, `my_`). Después hay que verificar 100 filas y cero nulos en el índice.

## 5. Consultas y extracción de datos (ID3.1, ID4.1)

*Pendiente: bloque de Yerko.* Registrar las cuatro vistas temporales (`createOrReplaceTempView`) y ejecutar cada consulta en una celda de Spark SQL separada:

- **C1–C3 y C7** usan `df_openmeteo`, con `fecha_hora` y `condicion_clima`. En C3 también se puede contrastar el conteo con la serie completa de `df_openmeteo_serie` (168 horas de la última semana).
- **C4 y C5** usan `df_aqicn`, con contaminantes en escala AQI. En C5, un valor de NO2 exactamente igual a 4 no cuenta.
- **C6** usa `df_mongo_bpm` con `altura_metros`.

## 6. Presentación

Video de 7 a 10 minutos (`mcdi502_s3_grupo1_video.mp4`), en el que participan los tres integrantes:

| Tiempo | Integrante | Contenido |
|---|---|---|
| 0:00–2:45 | Sebastián | Objetivo, arranque del entorno, conexiones a MongoDB y MySQL, conteos y conversión a metros |
| 2:45–5:25 | José | Conexiones a OpenMeteo y AQICN: serie horaria de 7 días, por qué no sirve el token `demo`, 100 estaciones más cercanas, escala AQI y resultados de `condicion_clima` |
| 5:25–8:30 | Yerko | Estrategia de integración y sus límites, vistas temporales y C1–C7 interpretadas |

## 7. Fuentes y atribuciones

- Notebooks base del docente (Semana 3, MCDI502): `-BaseEjemploOpenMeteo.ipynb`, `-BaseEjemploAQICN.ipynb`, `-BaseEjemploMongoDB.ipynb` y `-BaseEjemploMysqlPHP_APIRest.ipynb`.
- OpenMeteo, Weather Forecast API: https://open-meteo.com/en/docs
- World Air Quality Index Project, JSON API: https://aqicn.org/json-api/doc/ (datos usados con fines académicos, con atribución a la red de monitoreo de cada estación).
- PyMongo, consultas con orden y límite: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/specify-documents-to-return/
- Apache Spark 4, PySpark API: https://spark.apache.org/docs/latest/api/python/index.html